# 5 · The CPU samplers

Five samplers run on the base install. They take the same arguments and return the same
`SamplerResult`, so swapping one for another is a one-word change.

| sampler | what it is | reach for it when |
|---|---|---|
| `mcmc` | emcee ensemble | the sensible default |
| `nested` (`dynesty`) | nested sampling | you want **log-evidence** for a Bayes factor |
| `abc` | rejection ABC | likelihood-free; tolerant of awkward models |
| `abc_smc` | sequential ABC | ABC, but spending the simulations better |
| `snpe` / `npe` | neural simulation-based inference | amortised inference over many transients |

**Needs:** the base install. The `snpe` section needs the `[sbi]` extra.

> **Budgets here are illustrative, not publication-grade.** They are sized so the notebook runs in
> minutes. Before quoting any number, raise the budget until `result.info["converged"]` is `True`,
> and only compare fits that use the same `space=`.

In [1]:
from pathlib import Path

# Works whether the notebook is run from notebooks/ or from the repository root.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "tests" / "data").is_dir())
AT2017GFO = ROOT / "tests" / "data" / "at2017gfo.csv"

import warnings
warnings.filterwarnings("ignore")           # keep the tutorial output readable

import numpy as np
import matplotlib.pyplot as plt
import whisper_cbpf as wp

print("whisper_cbpf", wp.__version__)

whisper_cbpf 0.1.0


In [2]:
print(wp.list_samplers())

['abc', 'abc_gpu', 'abc_smc', 'abc_smc_gpu', 'dynesty', 'emcee_jax', 'mcmc', 'nested', 'npe', 'nuts_gpu', 'pymc_jax_gpu_parallelized', 'pymc_jax_gpu_vectorized', 'snpe', 'snpe_gpu']


## The data

In [3]:
lc = wp.load_lightcurve(AT2017GFO, redshift=0.0098)
lc = lc.select_bands(["g"]).set_explosion_date(57982.529).select_time_window(0.0, 10.0)
if "upper_limit" in lc.colnames:
    lc = lc.where(upper_limit=False)
print(lc.n_points, "points in g")

53 points in g


## One fit

`wp.fit(lc, model, sampler=...)` is the front door. Every sampler also has a named entry point
(`fit_MCMC`, `fit_nested`, …) if you prefer it.

In [4]:
r = wp.fit(lc, "bazin", sampler="mcmc", space="magnitude", nsteps=2000, burnin=500, seed=0)
print(r)

SamplerResult(sampler='mcmc', model='bazin', n_samples=2400, AIC=279.3, runtime=2.20s)


In [5]:
for name, st in r.summary.items():
    print(f"  {name:10s} {st['median']:10.4f}  [{st['ci16']:9.4f}, {st['ci84']:9.4f}]")
print()
print("AIC", round(r.aic, 2), " BIC", round(r.bic, 2), " WAIC", round(r.waic, 2))
print("converged:", r.info.get("converged"))

  amplitude      0.0001  [   0.0000,    1.8742]
  t0            12.5053  [  -7.3853,   28.7752]
  tau_rise      11.3469  [   4.3500,   14.5117]
  tau_fall       5.3562  [   0.9251,   41.9208]

AIC 279.34  BIC 287.22  WAIC 214149541.55
converged: False


### The result object

`SamplerResult` is the same shape for every sampler: the draws as a DataFrame, a per-parameter
summary, the best fit, and the sampler-specific diagnostics under `info`.

In [6]:
print("samples :", type(r.samples).__name__, r.samples.shape)
r.samples.head(3)

samples : DataFrame (2400, 4)


,amplitude,t0,tau_rise,tau_fall
0,0.029326,28.755836,4.016727,31.673999
1,1.820171,28.708727,2.086565,9.182871
2,0.016907,27.344834,1.952575,3.323312


In [7]:
print("diagnostics in info:")
for k, v in r.info.items():
    if k not in ("band_metrics", "predictive_metrics"):
        print(f"  {k:26s} {v}")

diagnostics in info:
  nwalkers                   16
  nsteps                     2000
  burnin                     500
  thin                       10
  space                      magnitude
  likelihood                 GaussianLikelihood
  mean_acceptance_fraction   0.0895
  mean_autocorr_time         190.45177057231322
  n_samples_per_walker       150
  converged                  False
  seed                       0


## Every CPU sampler on the same data

Same model, same space, same seed. The budgets are small, so read the spread as "these are the same
posterior to within a small-budget wobble", not as a ranking of samplers.

In [8]:
runs = {}
runs["mcmc"]    = wp.fit(lc, "bazin", sampler="mcmc", space="magnitude",
                         nsteps=2000, burnin=500, seed=0)
runs["nested"]  = wp.fit(lc, "bazin", sampler="nested", space="magnitude",
                         nlive=200, maxcall=40000, seed=0)
runs["abc"]     = wp.fit(lc, "bazin", sampler="abc", space="magnitude",
                         n_simulations=4000, quantile=0.05, seed=0)
runs["abc_smc"] = wp.fit(lc, "bazin", sampler="abc_smc", space="magnitude",
                         n_particles=200, n_rounds=4, seed=0)

print(f"{'sampler':10s} {'AIC':>10s} {'BIC':>10s} {'draws':>7s} {'runtime':>9s}  converged")
for name, res in runs.items():
    print(f"{name:10s} {res.aic:10.1f} {res.bic:10.1f} {res.n_samples:7d} "
          f"{res.runtime_s:8.1f}s  {res.info.get('converged')}")

sampler           AIC        BIC   draws   runtime  converged
mcmc            279.3      287.2    2400      1.4s  False
nested         1249.5     1257.4    1479     19.2s  False
abc            8521.4     8529.3     200      0.4s  None
abc_smc       37701.2    37709.1     200      1.0s  None


### Only `nested` gives you an evidence

`ln Z` is what a Bayes factor is built from. It is also the one number here that depends on how wide
you made your priors, in a way AIC and BIC do not — so only compare models whose priors you would
defend before seeing the data.

In [9]:
n = runs["nested"]
print("log evidence:", round(n.info["log_evidence"], 3), "+/-", round(n.info["log_evidence_err"], 3))

log evidence: -633.142 +/- 2.929


### The medians agree even where the information criteria do not

In [10]:
params = list(runs["mcmc"].summary)
print(f"{'param':10s}" + "".join(f"{k:>14s}" for k in runs))
for p in params:
    row = "".join(f"{runs[k].summary[p]['median']:14.4f}" if p in runs[k].summary else f"{'-':>14s}"
                  for k in runs)
    print(f"{p:10s}{row}")

param               mcmc        nested           abc       abc_smc
amplitude         0.0001        2.6670        3.7605        1.9547
t0               12.5053       -6.9266       20.7790       17.1633
tau_rise         11.3469       13.3703        2.2212        3.5156
tau_fall          5.3562        0.9009       35.4473       27.8299


## SNPE — amortised, neural

`snpe` trains a normalising flow on simulations, so the expensive part is the training and the
posterior is then cheap to query. `num_rounds=1` is amortised NPE; more rounds focus the training on
the observation you have.

In [11]:
try:
    s = wp.fit(lc, "bazin", sampler="snpe", space="magnitude",
               num_rounds=1, num_simulations=400, num_samples=500, seed=0)
    print(s)
    for name, st in s.summary.items():
        print(f"  {name:10s} {st['median']:10.4f}")
except ImportError as exc:
    print("needs the [sbi] extra:", exc)

2026-08-21 18:24:15.693650: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-21 18:24:15.715374: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787336655.740131  588084 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787336655.747798  588084 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787336655.767804  588084 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.


 Neural network successfully converged after 469 epochs.

SamplerResult(sampler='snpe', model='bazin', n_samples=500, AIC=7231.5, runtime=8.24s)
  amplitude      0.1669
  t0             1.0549
  tau_rise      14.4170
  tau_fall       8.8393


### The embedding network

SNPE conditions on a summary of the light curve. `build_embedding` gives you an MLP or a temporal
convolutional net to learn that summary instead of hand-designing one.

In [12]:
try:
    from whisper_cbpf.embeddings import build_embedding
    net = build_embedding("mlp", n_points=lc.n_points, latent_dim=16)
    print(net)
except ImportError as exc:
    print("needs torch:", exc)

MLPEmbedding(
  (net): Sequential(
    (0): Linear(in_features=53, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=16, bias=True)
  )
)


## Registering your own sampler

Subclass `BaseSampler`, implement `fit`, register it. `wp.fit(..., sampler="your_name")` then works.

In [13]:
from whisper_cbpf.samplers import BaseSampler, SamplerResult
from whisper_cbpf.samplers.base import aic_bic, summarize_posterior
import pandas as pd, time


class PriorOnlySampler(BaseSampler):
    """Draws from the prior and ignores the data -- a baseline, not an inference method."""

    name = "prior_only"

    def fit(self, lc, model, prior=None, *, n_draws=500, space="auto", seed=0, **kw):
        model = wp.get_model(model)
        prior = prior or model.default_prior
        rng = np.random.default_rng(seed)
        t0 = time.perf_counter()
        draws = pd.DataFrame([prior.sample(rng) for _ in range(n_draws)])
        k, n = len(model.parameters), lc.n_points
        aic, bic = aic_bic(np.nan, k, n)
        return SamplerResult(
            sampler=self.name, model=model.name, parameters=list(model.parameters),
            samples=draws, summary=summarize_posterior(draws, model.parameters),
            best_params=draws.iloc[0].to_dict(), n_data=n, n_params=k,
            runtime_s=time.perf_counter() - t0, info={"note": "prior draws, data ignored"},
            aic=aic, bic=bic)


wp.register_sampler("prior_only", PriorOnlySampler, overwrite=True)
print("prior_only" in wp.list_samplers())
print(wp.fit(lc, "bazin", sampler="prior_only", n_draws=200))

True
SamplerResult(sampler='prior_only', model='bazin', n_samples=200, AIC=nan, runtime=0.01s)


## Where next

* [06 · JAX physical models](06_jax_physical_models.ipynb) and
  [07 · GPU samplers](07_gpu_samplers.ipynb) — the same job, on a GPU.
* [08 · Metrics and model comparison](08_metrics_and_comparison.ipynb) — what to do with these
  numbers.
* [10 · Plotting](10_plotting.ipynb) — look at the fits above.